# 🌿 Plant Species Classification - Training Notebook
## Обучение нейросети для классификации 48 видов растений по листьям

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-username/plant-classification/blob/main/train.ipynb)

### 📌 Инструкция:
1. Нажмите `Runtime` → `Run all` (или запускайте ячейки по очереди)
2. Дождитесь окончания обучения (~30-60 минут на CPU)
3. Скачайте модель `best_plant_model.pt` из панели слева

---

## 1. Установка библиотек

In [ ]:
# Установка необходимых библиотек
!pip install ultralytics -q
!pip install kagglehub -q
!pip install seaborn -q
!pip install scikit-learn -q

print("✅ Все библиотеки установлены")

## 2. Скачивание датасета с Kaggle

In [ ]:
import kagglehub
import os

print("📥 Скачиваем датасет с 48 видами растений...")

# Скачиваем датасет
path = kagglehub.dataset_download("developerzulkarnain/48-plant-leaves-datasets")

# Путь к папке с изображениями
dataset_path = os.path.join(path, "Leaves of 48 Plants Dataset")

# Если структура другая, ищем папку
if not os.path.exists(dataset_path):
    for root, dirs, files in os.walk(path):
        if "Leaves of 48 Plants Dataset" in dirs:
            dataset_path = os.path.join(root, "Leaves of 48 Plants Dataset")
            break
        elif "48 Plant Leaves Dataset" in dirs:
            dataset_path = os.path.join(root, "48 Plant Leaves Dataset")
            break

print(f"✅ Датасет скачан в: {dataset_path}")

## 3. Анализ структуры датасета

In [ ]:
import numpy as np
from collections import defaultdict

# Получаем список всех классов (папок)
plant_classes = [d for d in os.listdir(dataset_path) 
                 if os.path.isdir(os.path.join(dataset_path, d))]
plant_classes.sort()

print(f"🌿 Найдено классов растений: {len(plant_classes)}")
print(f"📋 Первые 10 классов: {plant_classes[:10]}")

# Собираем все изображения по классам
class_to_images = defaultdict(list)

for plant_class in plant_classes:
    class_path = os.path.join(dataset_path, plant_class)
    for img_file in os.listdir(class_path):
        if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
            class_to_images[plant_class].append(os.path.join(class_path, img_file))

# Статистика
total_images = sum(len(imgs) for imgs in class_to_images.values())
class_sizes = {cls: len(imgs) for cls, imgs in class_to_images.items()}

print(f"\n📊 Статистика датасета:")
print(f"  Всего изображений: {total_images}")
print(f"  Среднее на класс: {total_images/len(plant_classes):.0f}")
print(f"  Максимум: {max(class_sizes.values())}")
print(f"  Минимум: {min(class_sizes.values())}")

## 4. Подготовка данных для YOLO

In [ ]:
import shutil
from sklearn.model_selection import train_test_split

# Создаем папки для данных YOLO
output_dir = '/content/plant_dataset_48'
os.makedirs(f'{output_dir}/train', exist_ok=True)
os.makedirs(f'{output_dir}/val', exist_ok=True)

print("📂 Создаем структуру данных...")

skipped_classes = []

for plant_class, images in class_to_images.items():
    if len(images) < 2:
        skipped_classes.append(plant_class)
        continue
    
    # Создаем папки для класса
    os.makedirs(f'{output_dir}/train/{plant_class}', exist_ok=True)
    os.makedirs(f'{output_dir}/val/{plant_class}', exist_ok=True)
    
    # Разделяем на train (80%) и val (20%)
    train_imgs, val_imgs = train_test_split(images, test_size=0.2, random_state=42)
    
    # Копируем файлы
    for img_path in train_imgs:
        dst = f'{output_dir}/train/{plant_class}/{os.path.basename(img_path)}'
        shutil.copy2(img_path, dst)
    
    for img_path in val_imgs:
        dst = f'{output_dir}/val/{plant_class}/{os.path.basename(img_path)}'
        shutil.copy2(img_path, dst)

if skipped_classes:
    print(f"⚠️ Пропущено классов (мало изображений): {skipped_classes}")

# Проверка результата
train_classes = len(os.listdir(f'{output_dir}/train'))
val_classes = len(os.listdir(f'{output_dir}/val'))
train_total = sum(len(os.listdir(f'{output_dir}/train/{cls}')) for cls in os.listdir(f'{output_dir}/train'))
val_total = sum(len(os.listdir(f'{output_dir}/val/{cls}')) for cls in os.listdir(f'{output_dir}/val'))

print(f"\n✅ Данные готовы:")
print(f"  Train: {train_total} изображений, {train_classes} классов")
print(f"  Val: {val_total} изображений, {val_classes} классов")

# Сохраняем список классов
with open('/content/plant_classes.txt', 'w') as f:
    for cls in plant_classes:
        f.write(f"{cls}\n")
print("💾 Список классов сохранен в plant_classes.txt")

## 5. Обучение модели YOLOv8

In [ ]:
import torch
from ultralytics import YOLO

# Определяем устройство
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️ Устройство: {device.upper()}")

if device == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Видеопамять: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Настройки обучения
EPOCHS = 50
BATCH_SIZE = 32 if device == 'cuda' else 16
IMAGE_SIZE = 224
PATIENCE = 15

print(f"\n📊 Параметры обучения:")
print(f"  Эпох: {EPOCHS}")
print(f"  Размер батча: {BATCH_SIZE}")
print(f"  Размер изображений: {IMAGE_SIZE}")

# Загружаем предобученную модель
model = YOLO('yolov8n-cls.pt')
print("\n✅ Модель загружена")

# Запускаем обучение
results = model.train(
    data=output_dir,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=device,
    workers=4,
    patience=PATIENCE,
    project='plant_classification',
    name='48_plants_full',
    exist_ok=True,
    pretrained=True,
    verbose=True,
    seed=42
)

print("\n✅ Обучение завершено!")

## 6. Валидация модели

In [ ]:
# Путь к лучшей модели
best_model_path = '/content/plant_classification/48_plants_full/weights/best.pt'

# Загружаем лучшую модель
best_model = YOLO(best_model_path)

# Валидация
metrics = best_model.val()

print(f"\n🎯 Результаты валидации:")
print(f"  Accuracy (Точность): {metrics.top1:.4f}")
print(f"  Top-5 Accuracy: {metrics.top5:.4f}")
print(f"  Loss: {metrics.loss:.4f}")

# Сохраняем метрики
with open('/content/training_metrics.txt', 'w') as f:
    f.write(f"Accuracy: {metrics.top1:.4f}\n")
    f.write(f"Top-5 Accuracy: {metrics.top5:.4f}\n")
    f.write(f"Loss: {metrics.loss:.4f}\n")
    f.write(f"Epochs: {EPOCHS}\n")
    f.write(f"Classes: {len(plant_classes)}\n")

print("\n✅ Метрики сохранены в training_metrics.txt")

## 7. Сохранение и скачивание модели

In [ ]:
# Копируем модель в корень для удобства
shutil.copy(best_model_path, '/content/best_plant_model.pt')
print("✅ Модель сохранена как best_plant_model.pt")

# Показываем графики обучения
from IPython.display import Image as IPImage

results_img = '/content/plant_classification/48_plants_full/results.png'
if os.path.exists(results_img):
    print("\n📈 Графики обучения:")
    display(IPImage(filename=results_img, width=800))

# Выводим информацию о файлах для скачивания
print("\n" + "="*60)
print("🎉 ОБУЧЕНИЕ УСПЕШНО ЗАВЕРШЕНО!")
print("="*60)

print("\n📁 Файлы для скачивания:")
print("  🤖 best_plant_model.pt - обученная модель")
print("  📋 plant_classes.txt - список 48 видов растений")
print("  📊 training_metrics.txt - метрики обучения")

print("\n⬇️ Как скачать файлы в Colab:")
print("  1. Откройте панель слева (📂 Файлы)")
print("  2. Найдите нужный файл")
print("  3. Нажмите на три точки → Скачать")

## 8. Тестирование модели (опционально)

In [ ]:
import random
from PIL import Image
import matplotlib.pyplot as plt

# Функция для тестирования
def test_random_image():
    # Выбираем случайное изображение из валидационной папки
    val_classes = os.listdir(f'{output_dir}/val')
    random_class = random.choice(val_classes)
    class_path = f'{output_dir}/val/{random_class}'
    random_img = random.choice(os.listdir(class_path))
    img_path = os.path.join(class_path, random_img)
    
    # Предсказание
    results = best_model(img_path)
    
    # Получаем результат
    predicted_class = results[0].names[results[0].probs.top1]
    confidence = results[0].probs.top1conf.item()
    
    # Отображаем
    img = Image.open(img_path)
    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.axis('off')
    
    color = 'green' if random_class == predicted_class else 'red'
    plt.title(f'Реальный: {random_class}\nПредсказанный: {predicted_class}\nУверенность: {confidence:.2%}', 
              color=color, fontsize=12)
    plt.show()
    
    return random_class, predicted_class, confidence

# Тестируем на 3 случайных изображениях
print("🧪 Тестирование на случайных изображениях:\n")
for i in range(3):
    print(f"Тест {i+1}:")
    real, pred, conf = test_random_image()
    print(f"  Результат: {'✅ Правильно' if real == pred else '❌ Ошибка'}\n")

print("\n💡 Используйте эту функцию для тестирования своих фото:")
print("  test_random_image() - случайное тестовое фото")
print("  best_model('/путь/к/вашему/фото.jpg') - ваше фото")

## 9. Использование модели

```python
# Пример использования обученной модели
from ultralytics import YOLO

# Загрузка модели
model = YOLO('best_plant_model.pt')

# Распознавание растения
results = model('путь_к_фото_листа.jpg')

# Результат
plant_name = results[0].names[results[0].probs.top1]
confidence = results[0].probs.top1conf.item()
print(f"🌿 Растение: {plant_name} (Уверенность: {confidence:.2%})")
```

---

### 📞 Помощь и поддержка

Если возникли проблемы:
1. Проверьте, что выбрано окружение `Runtime` → `Change runtime type` → `T4 GPU` (для ускорения)
2. Перезапустите среду выполнения: `Runtime` → `Restart runtime`
3. Создайте Issue на GitHub

---
⭐ **Если ноутбук оказался полезным, поставьте звезду на GitHub!**